### **Student Information**
GitHub ID: eon05

Kaggle name: emilyon

Kaggle private scoreboard snapshot: 

![pic_ranking.png](./pics/pic_ranking.png)

*Your name and student ID are deliberately not asked for here: this notebook goes into your repository, which is public. They belong in the git-ignored `config/.env`, where the other two notebooks read them from, and your NTU COOL submission is what ties the repository link back to you.*

---

# **Instructions**

This lab has **three parts**: exercises in the Master notebook, exercises and an agent conversation in the AgentPipeline notebook, and an internal Kaggle competition documented in this Homework notebook. Together they add up to **100 points**.

**Environment recommendations to solve lab 2:**
- **Part 1 and 2 exercises:** Everything can be run with only CPU.
- **Part 3 exercises:** For the competition you will probably need GPU to train your models, so it is recommended to use Colab or Kaggle if you don't have a laptop with a dedicated GPU.

## **Part 1 (40 pts):**

1. __Main Exercises (35 pts):__ Do the **take home exercises** from Sections: `1. Data Preparation` to `8. Information Retrieval and RAG`, in the DM2026-Lab2-Master Notebook. Total: `13 exercises`. Commit your code and submit the repository link to NTU Cool.

2. **Code Comments (5 pts):** **Tidy up the code in your notebook**. 

## **Part 2 (30 pts):**

1. **Agent Prompt (20 pts):** Interact with the Agent using appropriate prompts to guide it through and complete all the required tasks.

2. **Main Exercises (10 pts):** Do the remaining **take home exercises** from DM2026-Lab2-AgentPipeline Notebook. Total: `3 exercises required`. Commit your code and submit the repository link to NTU Cool.

## **Part 3 (30 pts):**

1. **Kaggle Competition Participation (15 pts):** Submit your predictions to the [Data Mining Class Kaggle Competition](https://www.kaggle.com/t/d99def2e62194b0d8f633267c32e6d1e) by **November 22**. Check Kaggle for the exact closing time. The competition concerns emotion recognition on Twitter; participation points use the Private Leaderboard ranking:
    - If your submission meets or exceeds the **baseline score (F1 score: 0.5)**, and there are no    significant issues with your report or code, you will automatically receive a base score of 5 points of the 15 points.

        The remaining 10 points will be determined by your ranking among participants who meet or exceed the baseline:

        |Ranking  |Additional Points  |
        | --- | --- |
        |Top 10% |+10 |
        |Top 20% |+9 |
        |Top 30% |+8 |
        |Top 40% |+7 |
        |Top 50% |+6 |
        |Top 60% |+5 |
        |Top 70% |+4 |
        |Top 80% |+3 |
        |Top 90% |+2 |
        |<Top 90% |+1 |

    - If your submission does not reach the baseline score, your grade will be lower than 5. The score will range from 0 to 4 points, depending on your Private Leaderboard ranking and the relative performance of students whose submissions are also below the baseline.

2. **Competition Report (15 pts)** A report section to be filled in inside this notebook in Markdown Format, we already provided you with the template below. You need to describe your work developing the model for the competition. The report should include a section describing briefly the following elements: 
* All of your AI prompts and what you decide do use.
* Your preprocessing steps.
* The feature engineering steps.
* Explanation of your model.

[Markdown Guide - Basic Syntax](https://www.markdownguide.org/basic-syntax/)

**`Things to note for Part 3:`**

* **The code used for the competition should be in this Jupyter Notebook File** `DM2026-Lab2-Homework.ipynb`.

* **Push the code used for the competition to your repository**.

* **The code should have a clear separation for the same sections of the report, AI prompts, preprocessing, feature engineering and model explanation. Briefly comment your code for easier understanding, we provide a template at the end of this notebook.**

* Save a screenshot of your Kaggle Private Leaderboard ranking as `pics/pic_ranking.png` and keep the image link in the Student Information cell at the top of this notebook.

The Kaggle competition closes on **November 22**. The three days afterward are for completing this notebook's Markdown report, checking your files, and submitting the lab by **November 25 at 11:59 pm (Wednesday)**.

Push your work to your repository, then submit its link and a ZIP on the corresponding NTU COOL assignment. The ZIP must contain `DM2026-Lab2-Master.ipynb`, `DM2026-Lab2-AgentPipeline.ipynb`, `DM2026-Lab2-Homework.ipynb`, both encrypted logs in `session_logs/`, and `pics/pic_ranking.png`. Name it with your student ID and your name. The two logs are git-ignored, so they are not in your repository and the ZIP is the only copy that reaches us: open it before uploading and check both are inside. If you miss them, you can upload them on NTU COOL until **November 26 at 11:59 pm**. See `README.md` for the exact file paths and setup steps.

---

Next you will find the template report with some simple markdown syntax explanations, use it to structure your content.

You can delete the syntax suggestions after you use them.

---

***

# **Project Report**

## 1. Model Development (10 pts Required)

**Describe briefly each section, you can add graphs/charts to support your explanations.**

### 1.1 Preprocessing Steps

- **Loading and merging:** The posts came from `final_posts.json`, the train/test split from
  `data_identification.csv`, and the labels from `emotion.csv`. I merged all three on `post_id`,
  which gave 3,360 labelled training posts and 1,440 unlabelled test posts. I merged on
  `post_id` because it is the only column the three files share, so it is what tells us which
  text belongs to which label and which set.

- **Text cleaning (for TF-IDF only):** I reused `clean_text` from the Master notebook:
  lowercase, remove URLs, remove @mentions, and squeeze extra spaces. Lowercasing makes sure
  "Angry" and "angry" are counted as the same word, and URLs and usernames are removed because
  they are noise that says nothing about emotion.

- **Raw text for the pretrained models:** The three pretrained emotion models were given the
  original `text`, not the cleaned version. These models learned from normal writing, where
  capital letters, exclamation points and emojis carry emotion, so cleaning would remove
  useful information.

- **Validation split:** I held out 20% of the training posts (672 posts), stratified by label
  with `random_state=42`, and trained on the other 80%. Every idea was scored on this split
  with macro F1 before I used a Kaggle submission. Stratifying keeps the same mix of emotions
  in both parts; without it, the 20% might get too few fear posts (fear has only 288 training
  posts) to score fairly. The split matched the leaderboard closely (e.g. 0.49 on validation
  vs. 0.4990 public), so I trusted it to compare ideas.

### 1.2 Feature Engineering Steps

- **Baseline features, TF-IDF:** unigrams and bigrams with `min_df=2`, fit on the training
  text only. I started with TF-IDF because it was the best representation in the Master
  notebook (0.862 macro F1 on the tweets), so it was the natural baseline to try first. On this
  dataset, however, it only reached 0.49 validation macro F1. The Master's tweets usually named
  their emotion outright (e.g. "furious", "terrified"), which is ideal for word counting, while
  these Kaggle posts are longer, Reddit-style, and often describe a situation without any
  emotion word, so there are fewer clear words for TF-IDF to latch onto.

- **Emotion probabilities from pretrained models:** Instead of word counts, I ran each post
  through three public emotion models and used their output probabilities as features:
  - `j-hartmann/emotion-english-distilroberta-base`: 7 probabilities (Ekman's 6 emotions + neutral)
  - `j-hartmann/emotion-english-roberta-large`: the same 7 emotions from a larger model
  - `SamLowe/roberta-base-go_emotions`: 28 fine-grained emotions (e.g. annoyance, nervousness)

  Together this gives 42 features per post. Each model's results were saved to a CSV cache so
  they only had to be computed once. I used these pretrained models because they were already
  trained on large amounts of emotion-labelled text, so they understand how emotions are
  expressed, even indirectly; 3,360 posts are not enough to learn that from scratch.

- **Tried and rejected:** combining TF-IDF with the probability features (scaled by a weight W
  from 0.5 to 8) gave 0.615-0.618, no real gain over the probabilities alone (0.614). This
  suggests TF-IDF added no new information: the pretrained models' probabilities already
  captured what the word counts knew, and more.

### 1.3 Explanation of Your Model

**Final model:** a Logistic Regression trained on the 42 probability features from section 1.2.
Each pretrained model has its own set of labels (7 or 28), while the competition has 6, so the
Logistic Regression learns how to translate their scores into this dataset's labels. For
example, a high "neutral" or "annoyance" score might often mean disgust or anger here.

| Approach | Validation macro F1 | Public LB |
|---|---|---|
| TF-IDF + Linear SVM | 0.490 | 0.4990 |
| Small emotion model alone (highest of 6 emotions, no training) | 0.572 | – |
| Small emotion model + LR (7 features) | 0.614 | 0.5851 |
| Small + large + LR (14) | 0.633 | 0.6002 |
| **Small + large + GoEmotions + LR (42), final** | **0.647** | **0.6291** |
| Fine-tuned RoBERTa-large (3 epochs, lr 1e-5, Kaggle GPU) | 0.645 | – |
| Blend: fine-tuned + 42-feature stack | 0.637-0.649 | – |
| Fine-tuned RoBERTa-large (4 epochs, lr 2e-5) | 0.636 | – |

No single pretrained model won on its own: as LR features, the small model alone reached 0.614,
the large model 0.605 and GoEmotions 0.595. Stacking them gave a small but consistent gain each
time a model was added (0.614 → 0.633 → 0.647), because different models make different
mistakes. Fine-tuning RoBERTa-large on our labels reached about the same level (0.645), and
blending it with the stack did not help, likely because it started from the same RoBERTa-large
model that was already in the stack, so it made similar mistakes.

**Final result:** on the private leaderboard, the final model scored 0.6413 macro F1, up from
0.6291 on the public leaderboard, and moved up 2 places to rank 41. The private score is close
to my validation estimate (0.647), which suggests the 80/20 validation split gave a reliable
picture of how the model would perform on unseen posts.

### 1.4 AI Prompts, what and why you decide to use that technique that suggest by AI

I used Claude (an AI assistant) as a tutor and for debugging throughout the competition. Every
suggestion was tested on my 80/20 validation split before I accepted it.

| What I asked | AI suggestion | Adopted / rejected, and why |
|---|---|---|
| How can I get above the 0.5 baseline? (TF-IDF + SVM scored 0.49) | `class_weight="balanced"` for the imbalanced classes | Rejected: 0.49 → 0.488, no real change |
| How can I get from 0.49 to around 0.66, to reach the top 10%? | Use a pretrained emotion model's probabilities as features | Adopted: 0.49 → 0.614 (public 0.5851) |
| What's the next improvement? | Combine TF-IDF with the probabilities | Rejected: 0.615-0.618 for every weight, no real gain over 0.614 |
| What else can push it higher? | Add the larger RoBERTa emotion model | Adopted: alone 0.605, but combined with the small model 0.633 (public 0.6002) |
| I'm ranked 47th of 64 — how can I move up for full credit? | Add GoEmotions (28 emotions) | Adopted: 0.647 (public 0.6291), my final submission |
| What could still get me closer to the top (0.6778) before the deadline? | Fine-tune RoBERTa-large on a Kaggle GPU | Rejected: 0.645 (3 epochs, lr 1e-5) and 0.636 (4 epochs, lr 2e-5), not better than 0.647 |
| Can the fine-tuned model and the stack be combined? | Blend the fine-tuned model with the stack | Rejected: 0.637-0.649 across blend weights, no real gain |

**Debugging help:** Kaggle rejected my first submission with "ID column id not found", because
the ID column had to be named `id`, not `post_id`. On Kaggle's GPU notebook, `TrainingArguments`
failed with "unexpected keyword argument 'warmup_ratio'", because the newer transformers library
had removed that option; I replaced it with `warmup_steps=50`.

Of the 7 suggestions, I adopted only 3 and rejected 4. This showed me that testing each idea
on a validation split, instead of simply trusting the AI's suggestion, is what actually led
to a better result, because most of the ideas that sounded promising did not improve the score.

---

**`From here on starts the code section for the competition.`**

---

# **Competition Code**

## 1. Preprocessing Steps

In [1]:
import pandas as pd

DATA_DIR = "data/kaggle/2026-dm-lab-2-private-competition/" # folder holding the 4 files

posts = pd.read_json(DATA_DIR + "final_posts.json")
print(posts.shape)
posts.head()

emotion = pd.read_csv(DATA_DIR + "emotion.csv")
print(emotion.shape)
display(emotion.head())

identification = pd.read_csv(DATA_DIR + "data_identification.csv")
print(identification.shape)
display(identification.head())

# attach train/test membership to every post
df = posts.merge(identification, on="post_id", how="left")

# attach labels (only training posts have one; test posts get NaN = empty)
df = df.merge(emotion, on="post_id", how="left")

print(df.shape)
display(df.head())
print(df["set"].value_counts())

train_df = df[df["set"] == "train"].copy() # rows to learn from
test_df  = df[df["set"] == "test"].copy() # rows to predict

print("train:", train_df.shape, "| test:", test_df.shape)
print("test labels all empty?", test_df["label"].isna().all())

(4800, 2)
(3360, 2)


,post_id,label
0,f8bv5zh,disgust
1,q0rr1l9,disgust
2,p94rwgk,sadness
3,9u6wtvm,fear
4,h8ea329,fear


(4800, 2)


,post_id,set
0,f8bv5zh,train
1,q0rr1l9,train
2,p94rwgk,train
3,9u6wtvm,train
4,h8ea329,train


(4800, 4)


,post_id,text,set,label
0,f8bv5zh,Thought it looked weird as well.,train,disgust
1,q0rr1l9,Bear died in a pit,train,disgust
2,p94rwgk,I have endured worse.,train,sadness
3,9u6wtvm,The pottery around Brasov is in danger,train,fear
4,h8ea329,"Meanwhile, our soldiers were approaching the a...",train,fear


set
train    3360
test     1440
Name: count, dtype: int64
train: (3360, 4) | test: (1440, 4)
test labels all empty? True


In [2]:
import re

# clean_text function from Master 2.1
def clean_text(s: str) -> str:
    s = s.lower()                             # Scared, SCARED -> scared
    s = re.sub(r"http\S+|www\.\S+", " ", s)   # URLs
    s = re.sub(r"@\w+", " ", s)               # @mentions
    return re.sub(r"\s+", " ", s).strip()     # squeeze the gaps the rules left

# New column, so the original "text" column survives for human reading.
train_df["clean"] = train_df["text"].map(clean_text)
test_df["clean"] = test_df["text"].map(clean_text)

# Check the rules by eye: print two tweets before and after.
for i in [1, 2]:
    print("RAW:  ", train_df["text"].iloc[i])
    print("CLEAN:", train_df["clean"].iloc[i])
    print()


train_df["clean"] = train_df["text"].map(clean_text)
test_df["clean"]  = test_df["text"].map(clean_text)

display(train_df[["text", "clean"]].head())

RAW:   Bear died in a pit
CLEAN: bear died in a pit

RAW:   I have endured worse.
CLEAN: i have endured worse.



,text,clean
0,Thought it looked weird as well.,thought it looked weird as well.
1,Bear died in a pit,bear died in a pit
2,I have endured worse.,i have endured worse.
3,The pottery around Brasov is in danger,the pottery around brasov is in danger
4,"Meanwhile, our soldiers were approaching the a...","meanwhile, our soldiers were approaching the a..."


## 2. Feature Engineering Steps

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score

# hold out 20% of the training posts to score ourselves
X_tr, X_val, y_tr, y_val = train_test_split(
    train_df["clean"], train_df["label"],
    test_size=0.2, random_state=42, stratify=train_df["label"])

# TF-IDF: fit on the 80% only, then transform the 20%
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
X_tr_vec  = vectorizer.fit_transform(X_tr)
X_val_vec = vectorizer.transform(X_val)

# train the model and score it
model = LinearSVC(random_state=42, class_weight="balanced")
model.fit(X_tr_vec, y_tr)
val_pred = model.predict(X_val_vec)
print("validation macro F1:", round(f1_score(y_val, val_pred, average="macro"), 3))

validation macro F1: 0.488


In [4]:
from transformers import pipeline
import os

# the pretrained emotion model: returns a score for all 7 emotions (top_k=None)
emo_model = pipeline("text-classification",
                     model="j-hartmann/emotion-english-distilroberta-base",
                     top_k=None, truncation=True)

CACHE = "data/kaggle/emotion_probs.csv" # saved results, so we only run this once

if os.path.exists(CACHE):
    emo_probs = pd.read_csv(CACHE)
else:
    outputs = emo_model(df["text"].tolist(), batch_size=32) # all 4,800 posts
    # each output is a list of 7 {label, score} dicts -> turn into one row per post
    emo_probs = pd.DataFrame([{d["label"]: d["score"] for d in row} for row in outputs])
    emo_probs.insert(0, "post_id", df["post_id"].values)
    emo_probs.to_csv(CACHE, index=False)

print(emo_probs.shape)
display(emo_probs.head())

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

(4800, 8)


,post_id,disgust,neutral,fear,surprise,sadness,anger,joy
0,f8bv5zh,0.839432,0.063344,0.031474,0.029712,0.015292,0.012980,0.007766
1,q0rr1l9,0.012462,0.025581,0.012270,0.010359,0.934344,0.003659,0.001324
2,p94rwgk,0.711715,0.020496,0.034182,0.001418,0.207037,0.023612,0.001541
3,9u6wtvm,0.002628,0.003608,0.979452,0.001288,0.005457,0.006838,0.000727
4,h8ea329,0.140495,0.011401,0.665292,0.004285,0.003060,0.173153,0.002313


In [5]:
EMO7 = ["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"]   # all model outputs
EMO6 = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]              # competition labels

# add the 7 probability columns to train and test, matched by post_id
train_df = train_df.merge(emo_probs, on="post_id", how="left")
test_df  = test_df.merge(emo_probs, on="post_id", how="left")
print(train_df.shape, test_df.shape)

(3360, 12) (1440, 12)


In [6]:
# do the validation rows picked by number still have the same labels as y_val?
print((train_df.loc[X_val.index, "label"].values == y_val.values).all())

True


In [7]:
emo_model_large = pipeline("text-classification",
                           model="j-hartmann/emotion-english-roberta-large",
                           top_k=None, truncation=True)

CACHE_LARGE = "data/kaggle/emotion_probs_large.csv"   # separate file, so the small model's results stay safe

if os.path.exists(CACHE_LARGE):
    emo_probs_large = pd.read_csv(CACHE_LARGE)   
else:
    outputs_large = emo_model_large(df["text"].tolist(), batch_size=16) # all 4,800 posts
    # each output is a list of 7 {label, score} dicts -> one row per post
    emo_probs_large = pd.DataFrame([{d["label"]: d["score"] for d in row} for row in outputs_large])
    emo_probs_large.insert(0, "post_id", df["post_id"].values)
    emo_probs_large.to_csv(CACHE_LARGE, index=False)

print(emo_probs_large.shape)
display(emo_probs_large.head())

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

(4800, 8)


,post_id,disgust,surprise,neutral,fear,sadness,anger,joy
0,f8bv5zh,0.784079,0.124519,0.049480,0.021700,0.010707,0.005944,0.003571
1,q0rr1l9,0.006688,0.022957,0.008038,0.528221,0.389099,0.043382,0.001615
2,p94rwgk,0.910745,0.000951,0.005422,0.004649,0.008467,0.069354,0.000411
3,9u6wtvm,0.014560,0.018850,0.045898,0.812061,0.014503,0.092828,0.001300
4,h8ea329,0.025292,0.004825,0.018695,0.787160,0.003704,0.159206,0.001118


In [8]:
# column names for the large model's outputs, marked with _L so they don't clash with the small model's
EMO7_L = [e + "_L" for e in EMO7]

# add the large model's 7 columns; suffixes=("", "_L") keeps the old names and adds _L to the new ones
train_df = train_df.merge(emo_probs_large, on="post_id", how="left", suffixes=("", "_L"))
test_df  = test_df.merge(emo_probs_large, on="post_id", how="left", suffixes=("", "_L"))
print(train_df.shape, test_df.shape)

# safety check: do the validation rows still line up?
print((train_df.loc[X_val.index, "label"].values == y_val.values).all())

(3360, 19) (1440, 19)
True


In [9]:
# GoEmotions model (RoBERTa trained on Reddit comments, 28 fine-grained emotions)
go_model = pipeline("text-classification",
                    model="SamLowe/roberta-base-go_emotions",
                    top_k=None, truncation=True)

CACHE_GO = "data/kaggle/emotion_probs_go.csv"   # separate cache file

if os.path.exists(CACHE_GO):
    emo_probs_go = pd.read_csv(CACHE_GO)         # already computed: just load it
else:
    outputs_go = go_model(df["text"].tolist(), batch_size=16)
    # prefix every label with "go_" so names like "anger" don't clash with the other models' columns
    emo_probs_go = pd.DataFrame([{"go_" + d["label"]: d["score"] for d in row} for row in outputs_go])
    emo_probs_go.insert(0, "post_id", df["post_id"].values)
    emo_probs_go.to_csv(CACHE_GO, index=False)

GO_COLS = [c for c in emo_probs_go.columns if c.startswith("go_")]   # the 28 emotion columns
print(emo_probs_go.shape, "|", len(GO_COLS), "GoEmotions columns")
display(emo_probs_go.head())

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

(4800, 29) | 28 GoEmotions columns


,post_id,go_neutral,go_disgust,go_annoyance,go_realization,go_embarrassment,go_disapproval,go_approval,go_fear,go_confusion,...,go_admiration,go_excitement,go_love,go_relief,go_grief,go_gratitude,go_joy,go_caring,go_remorse,go_pride
0,f8bv5zh,0.353961,0.338492,0.151596,0.082819,0.060904,0.036511,0.033177,0.026011,0.025380,...,0.002077,0.001616,0.001174,0.001055,0.000971,0.000964,0.000958,0.000884,0.000859,0.000682
1,q0rr1l9,0.314006,0.014492,0.011067,0.025155,0.004645,0.005147,0.010428,0.016102,0.002517,...,0.007398,0.001760,0.003839,0.001855,0.038573,0.001783,0.002928,0.003338,0.005941,0.000806
2,p94rwgk,0.027050,0.214598,0.050443,0.007021,0.008779,0.010429,0.008426,0.048357,0.001501,...,0.002998,0.001501,0.004906,0.001360,0.012400,0.002434,0.002355,0.005349,0.008338,0.000630
3,9u6wtvm,0.264422,0.003540,0.010183,0.011211,0.002054,0.014745,0.008515,0.260550,0.004824,...,0.002396,0.001535,0.001184,0.001956,0.004143,0.000833,0.001783,0.010251,0.003788,0.000509
4,h8ea329,0.848264,0.003760,0.005790,0.030290,0.001277,0.002189,0.090383,0.019170,0.001375,...,0.004826,0.003211,0.001307,0.001839,0.001088,0.000659,0.004531,0.003021,0.000784,0.001324


In [10]:
# add the 28 GoEmotions columns to train and test
train_df = train_df.merge(emo_probs_go, on="post_id", how="left")
test_df  = test_df.merge(emo_probs_go, on="post_id", how="left")
print(train_df.shape, test_df.shape)

# safety check: validation rows still line up?
print((train_df.loc[X_val.index, "label"].values == y_val.values).all())

(3360, 47) (1440, 47)
True


## 3. Model Implementation Steps

In [11]:
# fit TF-IDF and the model on ALL training posts
final_vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
X_train_all = final_vectorizer.fit_transform(train_df["clean"])    # learn + convert
X_test_all  = final_vectorizer.transform(test_df["clean"])     # convert only

final_model = LinearSVC(random_state=42)
final_model.fit(X_train_all, train_df["label"])

# predict the test posts
test_pred = final_model.predict(X_test_all)

# build the submission table in the sample file's format
submission = pd.DataFrame({"id": test_df["post_id"], "label": test_pred}) 
print(submission.shape)
display(submission.head())
print(submission["label"].value_counts())

submission.to_csv("submission_tfidf_svm.csv", index=False)

(1440, 2)


,id,label
0,xyzx9jp,anger
1,l3vzzxt,fear
2,g680iug,sadness
3,m4t8bp6,joy
4,ai3al6e,surprise


label
joy         433
anger       332
sadness     299
disgust     153
surprise    144
fear         79
Name: count, dtype: int64


In [12]:
zero_shot_pred = train_df[EMO6].idxmax(axis=1) # highest of the 6 competition emotions, per post
print("model alone, macro F1:", round(f1_score(train_df["label"], zero_shot_pred, average="macro"), 3))

model alone, macro F1: 0.572


In [13]:
from sklearn.linear_model import LogisticRegression

# the same 80/20 rows as Step 5
tr_rows  = train_df.loc[X_tr.index]
val_rows = train_df.loc[X_val.index]

prob_model = LogisticRegression(max_iter=1000)
prob_model.fit(tr_rows[EMO7], tr_rows["label"]) # learn from the 7 probability columns
prob_pred = prob_model.predict(val_rows[EMO7])
print("classifier on probabilities, macro F1:", round(f1_score(val_rows["label"], prob_pred, average="macro"), 3))

classifier on probabilities, macro F1: 0.614


In [14]:
# train the probability classifier on ALL 3,360 training posts
final_prob_model = LogisticRegression(max_iter=1000)
final_prob_model.fit(train_df[EMO7], train_df["label"])

# predict the 1,440 test posts
test_pred = final_prob_model.predict(test_df[EMO7])

submission = pd.DataFrame({"id": test_df["post_id"], "label": test_pred})
print(submission.shape)
print(submission["label"].value_counts())
submission.to_csv("submission_emoprobs_lr.csv", index=False)

(1440, 2)
label
joy         460
anger       291
sadness     227
surprise    175
disgust     175
fear        112
Name: count, dtype: int64


In [15]:
from scipy.sparse import hstack, csr_matrix

for W in [0.5, 1, 2, 4, 8]:
    # glue TF-IDF columns and the (weighted) 7 probability columns side by side
    X_tr_comb  = hstack([X_tr_vec,  csr_matrix(tr_rows[EMO7].values * W)]).tocsr()
    X_val_comb = hstack([X_val_vec, csr_matrix(val_rows[EMO7].values * W)]).tocsr()

    comb_model = LogisticRegression(max_iter=2000)
    comb_model.fit(X_tr_comb, y_tr)
    comb_pred = comb_model.predict(X_val_comb)
    print(f"W={W}: macro F1 =", round(f1_score(y_val, comb_pred, average="macro"), 3))

W=0.5: macro F1 = 0.616
W=1: macro F1 = 0.618
W=2: macro F1 = 0.616
W=4: macro F1 = 0.615
W=8: macro F1 = 0.615


In [16]:
tr_rows  = train_df.loc[X_tr.index]
val_rows = train_df.loc[X_val.index]

feature_sets = {
    "small model (7)":        EMO7,
    "large model (7)":        EMO7_L,
    "small + large (14)":     EMO7 + EMO7_L,
}

for name, cols in feature_sets.items():
    m = LogisticRegression(max_iter=2000)
    m.fit(tr_rows[cols], tr_rows["label"])
    pred = m.predict(val_rows[cols])
    print(f"{name}: macro F1 =", round(f1_score(val_rows["label"], pred, average="macro"), 3))

small model (7): macro F1 = 0.614
large model (7): macro F1 = 0.605
small + large (14): macro F1 = 0.633


In [17]:
# train on ALL 3,360 training posts using both models' probabilities (14 columns)
COLS = EMO7 + EMO7_L
final_combo_model = LogisticRegression(max_iter=2000)
final_combo_model.fit(train_df[COLS], train_df["label"])

test_pred = final_combo_model.predict(test_df[COLS])

submission = pd.DataFrame({"id": test_df["post_id"], "label": test_pred})
print(submission.shape)
print(submission["label"].value_counts())
submission.to_csv("submission_small_large_lr.csv", index=False)

(1440, 2)
label
joy         445
anger       322
sadness     231
surprise    178
disgust     154
fear        110
Name: count, dtype: int64


In [18]:
tr_rows  = train_df.loc[X_tr.index]
val_rows = train_df.loc[X_val.index]

feature_sets = {
    "small + large (14)":            EMO7 + EMO7_L,
    "GoEmotions alone (28)":         GO_COLS,
    "small + large + GoEmotions (42)": EMO7 + EMO7_L + GO_COLS,
}

for name, cols in feature_sets.items():
    m = LogisticRegression(max_iter=3000)
    m.fit(tr_rows[cols], tr_rows["label"])
    pred = m.predict(val_rows[cols])
    print(f"{name}: macro F1 =", round(f1_score(val_rows["label"], pred, average="macro"), 3))

small + large (14): macro F1 = 0.633
GoEmotions alone (28): macro F1 = 0.595
small + large + GoEmotions (42): macro F1 = 0.647


In [19]:
# train on ALL 3,360 training posts with all three models' probabilities (42 columns)
COLS = EMO7 + EMO7_L + GO_COLS
final_model_42 = LogisticRegression(max_iter=3000)
final_model_42.fit(train_df[COLS], train_df["label"])

test_pred = final_model_42.predict(test_df[COLS])

submission = pd.DataFrame({"id": test_df["post_id"], "label": test_pred})
print(submission.shape)
print(submission["label"].value_counts())
submission.to_csv("submission_three_models_lr.csv", index=False)

(1440, 2)
label
joy         434
anger       342
sadness     254
surprise    178
disgust     129
fear        103
Name: count, dtype: int64


In [20]:
# Step B: blend the 42-feature stack with the fine-tuned RoBERTa-large (from Kaggle GPU)

# 1) the 42-feature stack, trained on the 80%, gives probabilities for the 20%
COLS = EMO7 + EMO7_L + GO_COLS
stack_model = LogisticRegression(max_iter=3000).fit(tr_rows[COLS], tr_rows["label"])
stack_val = pd.DataFrame(stack_model.predict_proba(val_rows[COLS]),
                         columns=stack_model.classes_, index=val_rows["post_id"].values)

# 2) the fine-tuned model's probabilities for the same 20%, matched by post_id
ft_val = pd.read_csv("data/kaggle/ft_val_probs.csv").set_index("post_id")
ft_val = ft_val.loc[stack_val.index, stack_val.columns]     # same rows, same column order

# 3) try different mixes: w = how much weight the stack gets (1 - w goes to the fine-tuned model)
y_true = val_rows["label"].values
for w in [0, 0.25, 0.5, 0.75, 1]:
    blend = w * stack_val.values + (1 - w) * ft_val.values
    pred = stack_val.columns[blend.argmax(axis=1)]
    print(f"w={w}: macro F1 =", round(f1_score(y_true, pred, average="macro"), 3))

w=0: macro F1 = 0.645
w=0.25: macro F1 = 0.637
w=0.5: macro F1 = 0.646
w=0.75: macro F1 = 0.649
w=1: macro F1 = 0.647
